# ACLED Africa: weekly protest diffusion and riot escalation pipeline

**Research question:** How well can recent protest activity predict future protest occurrence over time and space, and do non-peaceful protests improve prediction of subsequent riots?

This notebook uses ACLED protest and riot events in Africa from 2010-2019. It builds a weekly 100 x 100 km grid-cell panel for predicting protest occurrence from recent focal and neighboring protest histories, plus an original country-week predictive baseline for protest-to-riot escalation. The revised no-country-restriction H3 inferential analysis is provided in `notebooks/acled_h1a_h1b_h2_h3_inferential_support.ipynb` and `scripts/inferential_tests_h1a_h1b_h2_h3.py`.

Hypotheses:
- **H1A Spatial diffusion:** recent protest activity in neighboring 100 x 100 km grid cells is expected to improve prediction of protest occurrence in the focal grid cell, beyond the focal cell's own recent protest history, measured over 1-week, 2-week, and 4-week windows.
- **H1B Temporal diffusion:** recent protest activity in the focal grid cell is expected to improve prediction of protest occurrence in the same grid cell.
- **H2 Peaceful versus non-peaceful diffusion:** recent peaceful protest histories are expected to have stronger predictive value for future protest occurrence than recent non-peaceful protest histories, both within the focal grid cell and in neighboring grid cells. Non-peaceful protests are defined as `Protest with intervention` and `Excessive force against protesters`.
- **H3 Protest escalation:** higher levels of non-peaceful protest events are expected to improve prediction of riots in the focal or neighboring 100 x 100 km grid cells in subsequent periods, without imposing a same-country restriction in the revised inferential specification.

Design choices:
- Weekly periods, with 4 weeks used as the 1-month history window.
- 100 x 100 km grid cells.
- Time split: 2010-2017 training, 2018-2019 test.
- Predictive models are used as evidence about clustering and temporal ordering, not as causal estimates.


## 0. Setup


In [ ]:
from pathlib import Path
import math
import os
import sys
import subprocess
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

try:
    from IPython.display import display, Image
except Exception:
    Image = None
    def display(x):
        print(x)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_recall_fscore_support, precision_recall_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

REPO_URL = "https://github.com/Mat99999/acled-spatiotemporal-conflict-model.git"
PROJECT_FOLDER_NAME = "acled_protests_riots_2010_2019_pipeline"
DATA_SUBDIR = Path("data") / "new data"
DATA_FILENAME = "ACLED_protests_riots_2010_2019.csv"


def running_in_colab():
    return "google.colab" in sys.modules


def find_project_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd / PROJECT_FOLDER_NAME, cwd.parent / PROJECT_FOLDER_NAME]
    for candidate in candidates:
        if (candidate / DATA_SUBDIR / DATA_FILENAME).exists():
            return candidate
    if running_in_colab() and REPO_URL:
        clone_dir = Path("/content") / PROJECT_FOLDER_NAME
        if not clone_dir.exists():
            subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
        return clone_dir
    raise FileNotFoundError(f"Could not find {DATA_SUBDIR / DATA_FILENAME}. Run from the project root or keep the notebook in notebooks/.")


PROJECT_DIR = find_project_dir()
DATA_PATH = PROJECT_DIR / DATA_SUBDIR / DATA_FILENAME
OUTPUT_DIR = PROJECT_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
PANELS_DIR = OUTPUT_DIR / "panels"
for directory in [TABLES_DIR, FIGURES_DIR, PANELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def save_and_show(fig, filename, dpi=140):
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=dpi)
    if Image is not None:
        display(Image(filename=str(path)))
    else:
        print(f"Saved figure: {path}")
    plt.close(fig)

RANDOM_STATE = 42
PRIMARY_GRID_KM = 100
TRAIN_END = pd.Timestamp('2017-12-31')
TEST_START = pd.Timestamp('2018-01-01')
TEST_END = pd.Timestamp('2019-12-31')
NONPEACEFUL_PROTEST_SUBTYPES = ['Protest with intervention', 'Excessive force against protesters']

print('Data:', DATA_PATH)
print('Project:', PROJECT_DIR)
print('Outputs:', OUTPUT_DIR)


## 1. Load and validate ACLED protests and riots


In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip().lower() for c in df.columns]

df['event_date'] = pd.to_datetime(df['event_date'], errors='coerce')
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df['fatalities'] = pd.to_numeric(df['fatalities'], errors='coerce').fillna(0)
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df['week'] = df['event_date'].dt.to_period('W-SUN').dt.start_time

df['is_protest'] = (df['event_type'] == 'Protests').astype(int)
df['is_riot'] = (df['event_type'] == 'Riots').astype(int)
df['peaceful_protest'] = ((df['event_type'] == 'Protests') & (df['sub_event_type'] == 'Peaceful protest')).astype(int)
df['nonpeaceful_protest'] = ((df['event_type'] == 'Protests') & (df['sub_event_type'].isin(NONPEACEFUL_PROTEST_SUBTYPES))).astype(int)

required = ['event_date','week','latitude','longitude','country','event_type','sub_event_type','fatalities']
print('Rows:', len(df))
print('Date range:', df['event_date'].min(), 'to', df['event_date'].max())
display(df[required].isna().sum().to_frame('missing'))
display(df.head(3))


In [ ]:
summary = pd.DataFrame({
    'metric': ['events','countries','weeks','protests','riots','peaceful_protests','nonpeaceful_protests','fatalities_total'],
    'value': [
        len(df), df['country'].nunique(), df['week'].nunique(),
        int(df['is_protest'].sum()), int(df['is_riot'].sum()),
        int(df['peaceful_protest'].sum()), int(df['nonpeaceful_protest'].sum()),
        int(df['fatalities'].sum()),
    ]
})
by_type = df['event_type'].value_counts().rename_axis('event_type').reset_index(name='events')
by_subtype = df['sub_event_type'].value_counts().rename_axis('sub_event_type').reset_index(name='events')
by_year = df.groupby(['year','event_type']).size().unstack(fill_value=0).reset_index()

display(summary)
display(by_type)
display(by_subtype)
display(by_year)
summary.to_csv(TABLES_DIR / '01_weekly_data_summary.csv', index=False)
by_type.to_csv(TABLES_DIR / '02_event_types.csv', index=False)
by_subtype.to_csv(TABLES_DIR / '03_sub_event_types.csv', index=False)
by_year.to_csv(TABLES_DIR / '04_events_by_year_type.csv', index=False)

weekly = df.groupby(['week','event_type']).size().reset_index(name='events')
fig, ax = plt.subplots(figsize=(12,4.5))
for event_type, sub in weekly.groupby('event_type'):
    ax.plot(sub['week'], sub['events'], linewidth=1.6, label=event_type)
ax.axvline(TEST_START, color='#B04A3A', linestyle='--', linewidth=1.2, label='Test starts')
ax.set_title('ACLED protests and riots per week, Africa 2010-2019')
ax.set_xlabel('Week')
ax.set_ylabel('Events')
ax.legend()
fig.tight_layout()
save_and_show(fig, 'weekly_events_by_type.png', dpi=160)


## 2. Build 100 km weekly grid-cell panel


In [ ]:
def add_grid_columns(events: pd.DataFrame, grid_km: int) -> pd.DataFrame:
    out = events.copy()
    lat0 = out['latitude'].median()
    km_per_degree_lat = 111.0
    km_per_degree_lon = 111.0 * math.cos(math.radians(lat0))
    out['x_km'] = out['longitude'] * km_per_degree_lon
    out['y_km'] = out['latitude'] * km_per_degree_lat
    out['grid_x'] = np.floor(out['x_km'] / grid_km).astype(int)
    out['grid_y'] = np.floor(out['y_km'] / grid_km).astype(int)
    out['cell_id'] = out['grid_x'].astype(str) + '_' + out['grid_y'].astype(str)
    out['grid_km'] = grid_km
    return out


def add_history_windows(panel: pd.DataFrame, group_col: str, columns, windows=(1,2,4)) -> pd.DataFrame:
    panel = panel.sort_values([group_col, 'week']).copy()
    for col in columns:
        for window in windows:
            panel[f'{col}_hist_{window}w'] = (
                panel.groupby(group_col)[col]
                .transform(lambda s: s.rolling(window, min_periods=1).sum())
                .fillna(0)
            )
    return panel


def build_cell_week_panel(events: pd.DataFrame, grid_km: int) -> pd.DataFrame:
    e = add_grid_columns(events.dropna(subset=['latitude','longitude','week']), grid_km)
    agg = e.groupby(['cell_id','grid_x','grid_y','week']).agg(
        protests=('is_protest','sum'),
        riots=('is_riot','sum'),
        peaceful_protests=('peaceful_protest','sum'),
        nonpeaceful_protests=('nonpeaceful_protest','sum'),
        fatalities=('fatalities','sum'),
        modal_country=('country', lambda s: s.mode().iat[0] if not s.mode().empty else np.nan),
    ).reset_index()

    cells = agg[['cell_id','grid_x','grid_y']].drop_duplicates()
    weeks = pd.date_range(agg['week'].min(), agg['week'].max(), freq='W-MON')
    panel = cells.merge(pd.DataFrame({'week': weeks}), how='cross')
    panel = panel.merge(agg, on=['cell_id','grid_x','grid_y','week'], how='left')
    count_cols = ['protests','riots','peaceful_protests','nonpeaceful_protests','fatalities']
    panel[count_cols] = panel[count_cols].fillna(0)
    panel['modal_country'] = panel.groupby('cell_id')['modal_country'].ffill().bfill()
    panel['protest_flag'] = (panel['protests'] > 0).astype(int)
    panel['riot_flag'] = (panel['riots'] > 0).astype(int)

    # Same-week neighbor counts are shifted into history windows below. The target is next week.
    base = panel[['grid_x','grid_y','week','protests','peaceful_protests','nonpeaceful_protests','riot_flag']].copy()
    shifted = []
    for dx in [-1,0,1]:
        for dy in [-1,0,1]:
            if dx == 0 and dy == 0:
                continue
            tmp = base.copy()
            tmp['grid_x'] = tmp['grid_x'] - dx
            tmp['grid_y'] = tmp['grid_y'] - dy
            shifted.append(tmp)
    neigh = pd.concat(shifted, ignore_index=True)
    neigh_agg = neigh.groupby(['grid_x','grid_y','week']).agg(
        neighbor_protests=('protests','sum'),
        neighbor_peaceful_protests=('peaceful_protests','sum'),
        neighbor_nonpeaceful_protests=('nonpeaceful_protests','sum'),
        neighbor_riot_cells=('riot_flag','sum'),
    ).reset_index()
    panel = panel.merge(neigh_agg, on=['grid_x','grid_y','week'], how='left')
    for col in ['neighbor_protests','neighbor_peaceful_protests','neighbor_nonpeaceful_protests','neighbor_riot_cells']:
        panel[col] = panel[col].fillna(0)

    panel = add_history_windows(
        panel,
        'cell_id',
        ['protests','peaceful_protests','nonpeaceful_protests','riots','neighbor_protests','neighbor_peaceful_protests','neighbor_nonpeaceful_protests'],
        windows=(1,2,4),
    )
    panel['target_protest_next_week'] = panel.groupby('cell_id')['protest_flag'].shift(-1)
    panel['target_riot_next_week'] = panel.groupby('cell_id')['riot_flag'].shift(-1)
    panel['week_of_year'] = panel['week'].dt.isocalendar().week.astype(int)
    panel['year'] = panel['week'].dt.year
    panel['sin_week'] = np.sin(2*np.pi*panel['week_of_year']/52)
    panel['cos_week'] = np.cos(2*np.pi*panel['week_of_year']/52)
    return panel.dropna(subset=['target_protest_next_week']).reset_index(drop=True)

panel100 = build_cell_week_panel(df, PRIMARY_GRID_KM)
print(panel100.shape)
print('Cells:', panel100['cell_id'].nunique(), '| weeks:', panel100['week'].nunique())
print('Target protest share:', round(panel100['target_protest_next_week'].mean(), 4))
display(panel100.head())
panel100.to_csv(PANELS_DIR / 'panel_100km_week.csv.gz', index=False, compression='gzip')


In [ ]:
panel_summary = pd.DataFrame({
    'metric': ['grid_km','cell_weeks','cells','weeks','target_protest_share','target_riot_share','mean_protests_when_positive'],
    'value': [
        PRIMARY_GRID_KM,
        len(panel100),
        panel100['cell_id'].nunique(),
        panel100['week'].nunique(),
        round(panel100['target_protest_next_week'].mean(), 4),
        round(panel100['target_riot_next_week'].mean(), 4),
        round(panel100.loc[panel100['protests'] > 0, 'protests'].mean(), 2),
    ]
})
display(panel_summary)
panel_summary.to_csv(TABLES_DIR / '05_panel_100km_week_summary.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12,4.2))
panel100.groupby('week')['protest_flag'].mean().plot(ax=axes[0], color='#2F5D8C')
axes[0].axvline(TEST_START, color='#B04A3A', linestyle='--', linewidth=1.2)
axes[0].set_title('Share of active protest cells by week')
axes[0].set_ylabel('Share with >=1 protest')
panel100.groupby('cell_id')['protests'].sum().clip(upper=150).hist(ax=axes[1], bins=35, color='#C8893D')
axes[1].set_title('Cell protest counts, clipped at 150')
axes[1].set_xlabel('Protests per cell, 2010-2019')
fig.tight_layout()
save_and_show(fig, 'panel_diagnostics_100km_week.png', dpi=160)


## 3. H1A/H1B weekly protest risk models

H1B is tested by comparing place/time features against focal-cell temporal history. H1A is tested by adding neighboring-cell protest histories after focal history is already included.


In [ ]:
BASE_FEATURES = ['grid_x','grid_y','year','sin_week','cos_week']
TEMPORAL_FEATURES = BASE_FEATURES + ['protests_hist_1w','protests_hist_2w','protests_hist_4w']
SPATIOTEMPORAL_FEATURES = TEMPORAL_FEATURES + ['neighbor_protests_hist_1w','neighbor_protests_hist_2w','neighbor_protests_hist_4w']
PEACEFUL_DIFFUSION_FEATURES = BASE_FEATURES + [
    'peaceful_protests_hist_1w','peaceful_protests_hist_2w','peaceful_protests_hist_4w',
    'nonpeaceful_protests_hist_1w','nonpeaceful_protests_hist_2w','nonpeaceful_protests_hist_4w',
    'neighbor_peaceful_protests_hist_1w','neighbor_peaceful_protests_hist_2w','neighbor_peaceful_protests_hist_4w',
    'neighbor_nonpeaceful_protests_hist_1w','neighbor_nonpeaceful_protests_hist_2w','neighbor_nonpeaceful_protests_hist_4w',
]
FEATURE_SETS = {
    'A_place_time': BASE_FEATURES,
    'B_temporal_focal_history': TEMPORAL_FEATURES,
    'C_spatiotemporal_neighbors': SPATIOTEMPORAL_FEATURES,
    'D_peaceful_nonpeaceful_diffusion': PEACEFUL_DIFFUSION_FEATURES,
}


def make_masks(panel):
    train = panel['week'] <= TRAIN_END
    test = (panel['week'] >= TEST_START) & (panel['week'] <= TEST_END)
    return train, test


def evaluate_predictions(y_true, y_prob, threshold=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    if threshold is None:
        threshold = y_true.mean()
    y_pred = (y_prob >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    return {
        'roc_auc': roc_auc_score(y_true, y_prob),
        'avg_precision': average_precision_score(y_true, y_prob),
        'brier': brier_score_loss(y_true, y_prob),
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'positive_rate_pred': y_pred.mean(),
    }


def fit_cell_models(panel, feature_sets):
    train_mask, test_mask = make_masks(panel)
    y_train = panel.loc[train_mask, 'target_protest_next_week'].astype(int)
    y_test = panel.loc[test_mask, 'target_protest_next_week'].astype(int)
    predictions = panel.loc[test_mask, ['cell_id','grid_x','grid_y','week','target_protest_next_week','protests','neighbor_protests']].copy()
    rows = []
    fitted = {}

    for name, features in feature_sets.items():
        features = [f for f in features if f in panel.columns]
        X_train = panel.loc[train_mask, features].fillna(0)
        X_test = panel.loc[test_mask, features].fillna(0)

        logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
        logit.fit(X_train, y_train)
        p_logit = logit.predict_proba(X_test)[:,1]
        row = {'model': f'Logit_{name}', 'feature_set': name, 'n_features': len(features)}
        row.update(evaluate_predictions(y_test, p_logit))
        rows.append(row)
        predictions[f'p_Logit_{name}'] = p_logit
        fitted[f'Logit_{name}'] = (logit, features)

        rf = RandomForestClassifier(n_estimators=80, min_samples_leaf=15, class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE)
        rf.fit(X_train, y_train)
        p_rf = rf.predict_proba(X_test)[:,1]
        row = {'model': f'RF_{name}', 'feature_set': name, 'n_features': len(features)}
        row.update(evaluate_predictions(y_test, p_rf))
        rows.append(row)
        predictions[f'p_RF_{name}'] = p_rf
        fitted[f'RF_{name}'] = (rf, features)

    return pd.DataFrame(rows).sort_values(['avg_precision','roc_auc'], ascending=False), predictions, fitted

train_mask, test_mask = make_masks(panel100)
print('Train rows:', train_mask.sum(), 'Test rows:', test_mask.sum())
print('Train target protest share:', round(panel100.loc[train_mask, 'target_protest_next_week'].mean(), 4))
print('Test target protest share:', round(panel100.loc[test_mask, 'target_protest_next_week'].mean(), 4))

cell_results, cell_predictions, fitted_cell_models = fit_cell_models(panel100, FEATURE_SETS)
display(cell_results)
cell_results.to_csv(TABLES_DIR / '06_h1_h2_cell_model_results_100km_week.csv', index=False)
cell_predictions.to_csv(PANELS_DIR / 'predictions_100km_week_test.csv', index=False)


In [ ]:
def model_row(results, model):
    return results.loc[results['model'] == model].iloc[0]

h1_rows = []
for family in ['Logit', 'RF']:
    a = model_row(cell_results, f'{family}_A_place_time')
    b = model_row(cell_results, f'{family}_B_temporal_focal_history')
    c = model_row(cell_results, f'{family}_C_spatiotemporal_neighbors')
    h1_rows.append({
        'model_family': family,
        'h1b_delta_ap_B_minus_A': b['avg_precision'] - a['avg_precision'],
        'h1b_delta_auc_B_minus_A': b['roc_auc'] - a['roc_auc'],
        'h1a_delta_ap_C_minus_B': c['avg_precision'] - b['avg_precision'],
        'h1a_delta_auc_C_minus_B': c['roc_auc'] - b['roc_auc'],
        'brier_A': a['brier'],
        'brier_B': b['brier'],
        'brier_C': c['brier'],
    })
h1_comparison = pd.DataFrame(h1_rows)
display(h1_comparison)
h1_comparison.to_csv(TABLES_DIR / '07_h1_temporal_spatial_uplift_100km_week.csv', index=False)

fig, ax = plt.subplots(figsize=(10,4.5))
plot_df = cell_results.sort_values('avg_precision', ascending=True)
ax.barh(plot_df['model'], plot_df['avg_precision'], color='#2F5D8C')
ax.set_title('Weekly protest-risk model comparison, 100 km grid')
ax.set_xlabel('Average precision on 2018-2019 test period')
ax.set_ylabel('')
fig.tight_layout()
save_and_show(fig, 'h1_h2_model_average_precision_100km_week.png', dpi=160)


## 4. H2 peaceful vs non-peaceful predictive value

This section uses the standardized logistic coefficients from the peaceful/non-peaceful prediction model. The comparison is descriptive: if peaceful protest histories have larger positive coefficients than non-peaceful histories, that is consistent with H2's predictive expectation.


In [ ]:
h2_model_name = 'Logit_D_peaceful_nonpeaceful_diffusion'
h2_model, h2_features = fitted_cell_models[h2_model_name]
h2_coef = pd.DataFrame({
    'feature': h2_features,
    'standardized_logit_coefficient': h2_model.named_steps['logisticregression'].coef_[0],
})
h2_coef['odds_ratio_per_1sd'] = np.exp(h2_coef['standardized_logit_coefficient'])
h2_focus = h2_coef[h2_coef['feature'].str.contains('peaceful_protests_hist|nonpeaceful_protests_hist')].copy()
h2_focus['scope'] = np.where(h2_focus['feature'].str.startswith('neighbor_'), 'spatial_neighbor_history', 'temporal_focal_history')
h2_focus['protest_type'] = np.where(h2_focus['feature'].str.contains('nonpeaceful'), 'nonpeaceful', 'peaceful')
h2_focus['window'] = h2_focus['feature'].str.extract(r'hist_(\d+w)')
display(h2_focus.sort_values(['scope','window','protest_type']))
h2_focus.to_csv(TABLES_DIR / '08_h2_peaceful_nonpeaceful_diffusion_coefficients.csv', index=False)

fig, ax = plt.subplots(figsize=(10,4.8))
plot_h2 = h2_focus.copy()
plot_h2['label'] = plot_h2['scope'].str.replace('_', ' ') + ' | ' + plot_h2['window'] + ' | ' + plot_h2['protest_type']
plot_h2 = plot_h2.sort_values('standardized_logit_coefficient')
colors = np.where(plot_h2['protest_type'] == 'peaceful', '#2F5D8C', '#B04A3A')
ax.barh(plot_h2['label'], plot_h2['standardized_logit_coefficient'], color=colors)
ax.axvline(0, color='#555555', linewidth=1)
ax.set_title('H2 coefficients: peaceful vs non-peaceful protest histories')
ax.set_xlabel('Standardized logistic coefficient')
fig.tight_layout()
save_and_show(fig, 'h2_peaceful_nonpeaceful_coefficients.png', dpi=160)


## 5. Ranking diagnostics for protest risk


In [ ]:
def threshold_sweep(y_true, y_prob, n_grid=501):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    rows = []
    for threshold in np.linspace(0, 1, n_grid):
        y_pred = (y_prob >= threshold).astype(int)
        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
        rows.append({'threshold': threshold, 'precision': precision, 'recall': recall, 'f1': f1, 'positive_rate_pred': y_pred.mean()})
    return pd.DataFrame(rows)


def precision_at_top_k(y_true, y_prob, top_shares=(0.01,0.05,0.10)):
    d = pd.DataFrame({'y_true': np.asarray(y_true).astype(int), 'y_prob': np.asarray(y_prob)})
    d = d.sort_values('y_prob', ascending=False).reset_index(drop=True)
    rows = []
    for share in top_shares:
        k = max(1, int(np.ceil(len(d) * share)))
        top = d.iloc[:k]
        rows.append({
            'top_share': share,
            'n_flagged': k,
            'precision_at_top': top['y_true'].mean(),
            'recall_at_top': top['y_true'].sum() / d['y_true'].sum(),
            'lift_vs_base_rate': top['y_true'].mean() / d['y_true'].mean(),
        })
    return pd.DataFrame(rows)

selected_cols = {
    'RF_B_temporal_focal_history': 'p_RF_B_temporal_focal_history',
    'RF_C_spatiotemporal_neighbors': 'p_RF_C_spatiotemporal_neighbors',
    'RF_D_peaceful_nonpeaceful_diffusion': 'p_RF_D_peaceful_nonpeaceful_diffusion',
    'Logit_C_spatiotemporal_neighbors': 'p_Logit_C_spatiotemporal_neighbors',
    'Logit_D_peaceful_nonpeaceful_diffusion': 'p_Logit_D_peaceful_nonpeaceful_diffusion',
}
selected_cols = {k:v for k,v in selected_cols.items() if v in cell_predictions.columns}
y_test = cell_predictions['target_protest_next_week'].astype(int)

threshold_rows = []
topk_rows = []
for model_name, col in selected_cols.items():
    sweep = threshold_sweep(y_test, cell_predictions[col])
    best = sweep.loc[sweep['f1'].idxmax()].copy()
    best['model'] = model_name
    best['base_rate'] = y_test.mean()
    threshold_rows.append(best)
    topk = precision_at_top_k(y_test, cell_predictions[col])
    topk.insert(0, 'model', model_name)
    topk_rows.append(topk)

threshold_eval = pd.DataFrame(threshold_rows)[['model','base_rate','threshold','precision','recall','f1','positive_rate_pred']]
topk_eval = pd.concat(topk_rows, ignore_index=True)
display(threshold_eval)
display(topk_eval)
threshold_eval.to_csv(TABLES_DIR / '09_f1_optimal_thresholds_100km_week.csv', index=False)
topk_eval.to_csv(TABLES_DIR / '10_precision_at_top_k_100km_week.csv', index=False)

fig, ax = plt.subplots(figsize=(8,5))
base_rate = y_test.mean()
ax.axhline(base_rate, color='#777777', linestyle='--', linewidth=1.2, label=f'Base rate ({base_rate:.3f})')
for model_name, col in selected_cols.items():
    precision, recall, _ = precision_recall_curve(y_test, cell_predictions[col])
    idx = np.unique(np.linspace(0, len(recall)-1, min(600, len(recall))).astype(int))
    ax.plot(recall[idx], precision[idx], linewidth=1.6, label=model_name)
ax.set_title('Precision-recall curves for weekly protest risk')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(fontsize=7)
fig.tight_layout()
save_and_show(fig, 'precision_recall_curves_100km_week.png', dpi=140)


## 6. Original H3 country-week riot escalation predictive baseline

This section keeps the original country-week predictive baseline for protest-to-riot escalation. The revised inferential H3 analysis removes the same-country restriction and tests whether non-peaceful protest activity predicts subsequent riots in the focal or neighboring 100 x 100 km grid cells; see the companion inferential notebook and script.


In [ ]:
def build_country_week_panel(events: pd.DataFrame) -> pd.DataFrame:
    agg = events.groupby(['country','week']).agg(
        protests=('is_protest','sum'),
        peaceful_protests=('peaceful_protest','sum'),
        nonpeaceful_protests=('nonpeaceful_protest','sum'),
        riots=('is_riot','sum'),
        fatalities=('fatalities','sum'),
    ).reset_index()
    countries = agg[['country']].drop_duplicates()
    weeks = pd.date_range(agg['week'].min(), agg['week'].max(), freq='W-MON')
    panel = countries.merge(pd.DataFrame({'week': weeks}), how='cross')
    panel = panel.merge(agg, on=['country','week'], how='left')
    count_cols = ['protests','peaceful_protests','nonpeaceful_protests','riots','fatalities']
    panel[count_cols] = panel[count_cols].fillna(0)
    panel['riot_flag'] = (panel['riots'] > 0).astype(int)
    panel['protest_flag'] = (panel['protests'] > 0).astype(int)
    panel = add_history_windows(panel, 'country', ['riots','protests','peaceful_protests','nonpeaceful_protests'], windows=(1,2,4))
    panel['target_riot_next_week'] = panel.groupby('country')['riot_flag'].shift(-1)
    panel['week_of_year'] = panel['week'].dt.isocalendar().week.astype(int)
    panel['year'] = panel['week'].dt.year
    panel['sin_week'] = np.sin(2*np.pi*panel['week_of_year']/52)
    panel['cos_week'] = np.cos(2*np.pi*panel['week_of_year']/52)
    return panel.dropna(subset=['target_riot_next_week']).reset_index(drop=True)

country_panel = build_country_week_panel(df)
print(country_panel.shape)
print('Countries:', country_panel['country'].nunique(), '| weeks:', country_panel['week'].nunique())
print('Target riot share:', round(country_panel['target_riot_next_week'].mean(), 4))
display(country_panel.head())
country_panel.to_csv(PANELS_DIR / 'country_week_riot_escalation_panel.csv', index=False)


In [ ]:
def fit_country_logit(panel, feature_sets):
    train_mask, test_mask = make_masks(panel)
    y_train = panel.loc[train_mask, 'target_riot_next_week'].astype(int)
    y_test = panel.loc[test_mask, 'target_riot_next_week'].astype(int)
    predictions = panel.loc[test_mask, ['country','week','target_riot_next_week','riots','nonpeaceful_protests','peaceful_protests']].copy()
    rows = []
    fitted = {}
    columns = {}

    for name, features in feature_sets.items():
        X_all = panel[features + ['country']].copy()
        X_all = pd.get_dummies(X_all, columns=['country'], drop_first=True)
        X_train = X_all.loc[train_mask].fillna(0)
        X_test = X_all.loc[test_mask].fillna(0)
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
        model.fit(X_train, y_train)
        probs = model.predict_proba(X_test)[:,1]
        row = {'model': f'Logit_{name}', 'feature_set': name, 'n_features': X_train.shape[1]}
        row.update(evaluate_predictions(y_test, probs))
        rows.append(row)
        predictions[f'p_Logit_{name}'] = probs
        fitted[f'Logit_{name}'] = model
        columns[f'Logit_{name}'] = list(X_train.columns)
    return pd.DataFrame(rows).sort_values(['avg_precision','roc_auc'], ascending=False), predictions, fitted, columns

H3_BASE_FEATURES = ['year','sin_week','cos_week','riots_hist_1w','riots_hist_2w','riots_hist_4w']
H3_PROTEST_FEATURES = H3_BASE_FEATURES + ['protests_hist_1w','protests_hist_2w','protests_hist_4w']
H3_ESCALATION_FEATURES = H3_BASE_FEATURES + [
    'peaceful_protests_hist_1w','peaceful_protests_hist_2w','peaceful_protests_hist_4w',
    'nonpeaceful_protests_hist_1w','nonpeaceful_protests_hist_2w','nonpeaceful_protests_hist_4w',
]
H3_FEATURE_SETS = {
    'A_riot_history': H3_BASE_FEATURES,
    'B_total_protest_history': H3_PROTEST_FEATURES,
    'C_peaceful_nonpeaceful_history': H3_ESCALATION_FEATURES,
}

h3_results, h3_predictions, h3_fitted, h3_columns = fit_country_logit(country_panel, H3_FEATURE_SETS)
display(h3_results)
h3_results.to_csv(TABLES_DIR / '11_h3_country_week_riot_model_results.csv', index=False)
h3_predictions.to_csv(PANELS_DIR / 'country_week_riot_predictions_test.csv', index=False)

h3_model_name = 'Logit_C_peaceful_nonpeaceful_history'
h3_model = h3_fitted[h3_model_name]
h3_cols = h3_columns[h3_model_name]
h3_coef = pd.DataFrame({
    'feature': h3_cols,
    'standardized_logit_coefficient': h3_model.named_steps['logisticregression'].coef_[0],
})
h3_coef['odds_ratio_per_1sd'] = np.exp(h3_coef['standardized_logit_coefficient'])
h3_focus = h3_coef[h3_coef['feature'].str.contains('peaceful_protests_hist|nonpeaceful_protests_hist')].copy()
h3_focus['protest_type'] = np.where(h3_focus['feature'].str.contains('nonpeaceful'), 'nonpeaceful', 'peaceful')
h3_focus['window'] = h3_focus['feature'].str.extract(r'hist_(\d+w)')
display(h3_focus.sort_values(['window','protest_type']))
h3_focus.to_csv(TABLES_DIR / '12_h3_escalation_coefficients.csv', index=False)

fig, ax = plt.subplots(figsize=(8,4.5))
plot_h3 = h3_focus.sort_values('standardized_logit_coefficient')
colors = np.where(plot_h3['protest_type'] == 'peaceful', '#2F5D8C', '#B04A3A')
ax.barh(plot_h3['feature'], plot_h3['standardized_logit_coefficient'], color=colors)
ax.axvline(0, color='#555555', linewidth=1)
ax.set_title('H3 coefficients: protest histories predicting next-week riots')
ax.set_xlabel('Standardized logistic coefficient')
fig.tight_layout()
save_and_show(fig, 'h3_riot_escalation_coefficients.png', dpi=160)


## 7. Brief notebook conclusion


In [ ]:
print('Key output tables:')
for name in [
    '01_weekly_data_summary.csv',
    '05_panel_100km_week_summary.csv',
    '06_h1_h2_cell_model_results_100km_week.csv',
    '07_h1_temporal_spatial_uplift_100km_week.csv',
    '08_h2_peaceful_nonpeaceful_diffusion_coefficients.csv',
    '11_h3_country_week_riot_model_results.csv',
    '12_h3_escalation_coefficients.csv',
]:
    print('-', TABLES_DIR / name)
